<a href="https://colab.research.google.com/github/Froznrobt/sudo-proxy/blob/dev/014varthini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install biopython requests

In [ ]:
from Bio import Entrez, SeqIO
import requests
import time


# ============================================================
# Configuration
# ============================================================

Entrez.email = "dvarthinius.bt24@rvce.edu.in"

GENE = "BRCA1"
ORGANISM = "Homo sapiens"

NCBI_ACCESSION = "NC_000017.11"
DDBJ_ACCESSION = "PC456323"

SEQUENCE_LENGTH = 100

SEARCH_QUERY = (
    f"{GENE}[gene] AND "
    f"{ORGANISM}[organism] AND "
    "mRNA[Title]"
)


# ============================================================
# 1. Search NCBI for BRCA1 mRNA
# ============================================================

def search_ncbi():
    print("\n========== NCBI Nucleotide ==========")
    print("Search query:", SEARCH_QUERY)

    handle = Entrez.esearch(
        db="nucleotide",
        term=SEARCH_QUERY,
        retmax=20,
        retmode="xml"
    )

    result = Entrez.read(handle)
    handle.close()

    ids = result["IdList"]
    count = int(result["Count"])

    print("Total NCBI records found:", count)
    print("First IDs:", ids[:5])

    return ids


# ============================================================
# 2. Fetch the first 100 bp of BRCA1 mRNA records
# ============================================================

def fetch_ncbi_mrna_records(ids):
    if not ids:
        print("No NCBI BRCA1 mRNA records found.")
        return []

    print("\nFetching NCBI BRCA1 mRNA records...")

    handle = Entrez.efetch(
        db="nucleotide",
        id=",".join(ids),
        rettype="fasta",
        retmode="text"
    )

    records = list(SeqIO.parse(handle, "fasta"))
    handle.close()

    mrna_information = []

    for record in records:
        first_100_bp = str(record.seq[:SEQUENCE_LENGTH])

        mrna_information.append({
            "database": "NCBI",
            "accession": record.id,
            "description": record.description,
            "length_retrieved": len(first_100_bp),
            "sequence": first_100_bp
        })

    return mrna_information


# ============================================================
# 3. Fetch the first 100 bp from NC_000017.11
# ============================================================

def fetch_ncbi_chromosome(accession):
    print("\n========== NCBI Chromosome ==========")
    print("Accession:", accession)

    try:
        handle = Entrez.efetch(
            db="nucleotide",
            id=accession,
            rettype="fasta",
            retmode="text",
            seq_start=1,
            seq_stop=SEQUENCE_LENGTH
        )

        sequence = handle.read()
        handle.close()

        filename = "NC_000017.11_first_100_bp.fasta"

        with open(filename, "w") as file:
            file.write(sequence)

        print("Saved as:", filename)
        print(sequence)

    except Exception as error:
        print("NCBI chromosome request failed:", error)


# ============================================================
# 4. Fetch the first 100 bp from DDBJ
# ============================================================

def fetch_ddbj(accession):
    print("\n========== DDBJ ==========")
    print("Accession:", accession)

    url = f"https://getentry.ddbj.nig.ac.jp/getentry/na/{accession}"

    parameters = {
        "format": "fasta",
        "start": 1,
        "end": SEQUENCE_LENGTH
    }

    try:
        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )

        print("DDBJ HTTP status:", response.status_code)

        response.raise_for_status()

        if response.text.strip():
            filename = f"{accession}_first_100_bp.fasta"

            with open(filename, "w") as file:
                file.write(response.text)

            print("Saved as:", filename)
            print(response.text)

        else:
            print("DDBJ returned an empty response.")

    except requests.RequestException as error:
        print("DDBJ request failed:", error)


# ============================================================
# 5. Search Europe PMC for BRCA1 publications
# ============================================================

def search_europe_pmc():
    print("\n========== Europe PMC ==========")

    query = '"BRCA1" AND "Homo sapiens"'

    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

    parameters = {
        "query": query,
        "format": "json",
        "pageSize": 10,
        "resultType": "core"
    }

    try:
        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()
        results = data.get("resultList", {}).get("result", [])

        print("Europe PMC results found:", data.get("hitCount", 0))

        for article in results:
            print("\nTitle:", article.get("title", "No title"))
            print("PMID:", article.get("pmid", "N/A"))
            print("DOI:", article.get("doi", "N/A"))

    except requests.RequestException as error:
        print("Europe PMC request failed:", error)


# ============================================================
# 6. Main program
# ============================================================

def main():
    # Search for BRCA1 mRNA records in NCBI
    ncbi_ids = search_ncbi()
    time.sleep(0.4)

    # Retrieve only the first 100 bp of each mRNA
    ncbi_mrna_information = fetch_ncbi_mrna_records(ncbi_ids)

    print("\nNCBI BRCA1 mRNA: first 100 bp")
    print("-" * 70)

    for item in ncbi_mrna_information:
        print("Accession:", item["accession"])
        print("Length retrieved:", item["length_retrieved"], "bp")
        print("Sequence:", item["sequence"])
        print("-" * 70)

    # Retrieve first 100 bp from the NCBI chromosome accession
    fetch_ncbi_chromosome(NCBI_ACCESSION)

    # Retrieve first 100 bp from DDBJ accession
    fetch_ddbj(DDBJ_ACCESSION)

    # Search Europe PMC for related articles
    search_europe_pmc()


if __name__ == "__main__":
    main()


========== NCBI Nucleotide ==========
Search query: BRCA1[gene] AND Homo sapiens[organism] AND mRNA[Title]
Total NCBI records found: 594
First IDs: ['1732746264', '1677529721', '1677501432', '1677500831', '2255456859']

Fetching NCBI BRCA1 mRNA records...

NCBI BRCA1 mRNA: first 100 bp
----------------------------------------------------------------------
Accession: NM_007294.4
Length retrieved: 100 bp
Sequence: GCTGAGACTTCCTGGACGGGGGACAGGCTGTGGGGTTTCTCAGATAACTGGGCCCCTGCGCTCAGGAGGCCTTCACCCTCTGCTCTGGGTAAAGTTCATT
----------------------------------------------------------------------
Accession: NM_007300.4
Length retrieved: 100 bp
Sequence: GCTGAGACTTCCTGGACGGGGGACAGGCTGTGGGGTTTCTCAGATAACTGGGCCCCTGCGCTCAGGAGGCCTTCACCCTCTGCTCTGGGTAAAGTTCATT
----------------------------------------------------------------------
Accession: NM_007299.4
Length retrieved: 100 bp
Sequence: GCTGAGACTTCCTGGACGGGGGACAGGCTGTGGGGTTTCTCAGATAACTGGGCCCCTGCGCTCAGGAGGCCTTCACCCTCTGCTCTGGTTCATTGGAACA
----------------------

In [ ]:
import requests

# ============================================================
# DDBJ configuration
# ============================================================

ACCESSION = "PC456323"

FASTA_FILE = "PC456323.fasta"
ANNOTATED_FILE = "PC456323.txt"

DDBJ_URL = (
    f"https://getentry.ddbj.nig.ac.jp/getentry/na/{ACCESSION}"
)


# ============================================================
# Fetch a record from DDBJ
# ============================================================

def fetch_ddbj(accession, output_format):
    """
    Fetch a DDBJ nucleotide record.

    output_format:
        fasta - nucleotide sequence
        flat  - annotated flat-file record
    """

    url = f"https://getentry.ddbj.nig.ac.jp/getentry/na/{accession}"

    parameters = {
        "format": output_format
    }

    try:
        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )

        print("Request URL:", response.url)
        print("HTTP status:", response.status_code)

        response.raise_for_status()

        if not response.text.strip():
            print("DDBJ returned an empty response.")
            return None

        return response.text

    except requests.exceptions.RequestException as error:
        print("Error while contacting DDBJ:", error)
        return None


# ============================================================
# Main program
# ============================================================

def main():
    print(f"Retrieving DDBJ accession: {ACCESSION}\n")

    # Retrieve sequence in FASTA format
    fasta_data = fetch_ddbj(ACCESSION, "fasta")

    if fasta_data:
        with open(FASTA_FILE, "w", encoding="utf-8") as file:
            file.write(fasta_data)

        print(f"\nFASTA sequence saved as: {FASTA_FILE}")
        print("\nFASTA preview:")
        print(fasta_data[:300])

    # Retrieve annotated DDBJ record
    annotated_data = fetch_ddbj(ACCESSION, "flat")

    if annotated_data:
        with open(ANNOTATED_FILE, "w", encoding="utf-8") as file:
            file.write(annotated_data)

        print(f"\nAnnotated record saved as: {ANNOTATED_FILE}")
        print("\nAnnotation preview:")
        print(annotated_data[:500])


if __name__ == "__main__":
    main()

Retrieving DDBJ accession: PC456323

Request URL: https://getentry.ddbj.nig.ac.jp/getentry/na/PC456323?format=fasta
HTTP status: 200

FASTA sequence saved as: PC456323.fasta

FASTA preview:
>PC456323|PC456323.1 JP 2021529838-A/1: METHODS AND COMPOSITIONS FOR MODULATING MYELOPEROXIDASE (MPO) EXPRESSION.
gacaatatcaggtgagctgtggaggtggggtccttggaagctggatgacagcagctggca
aggggataagagagcagtgagcccctccctcaaggaggtctggctttatccatagacagg
gccctctgaggtggggctgaggtacaaagggggattgagcagcccaggagaagagagatg
ggg
Request URL: https://getentry.ddbj.nig.ac.jp/getentry/na/PC456323?format=flat
HTTP status: 200

Annotated record saved as: PC456323.txt

Annotation preview:
LOCUS       PC456323               11080 bp    DNA     linear   PAT 29-JAN-2025
DEFINITION  JP 2021529838-A/1: METHODS AND COMPOSITIONS FOR MODULATING
            MYELOPEROXIDASE (MPO) EXPRESSION.
ACCESSION   PC456323
VERSION     PC456323.1
KEYWORDS    JP 2021529838-A/1.
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota;